In [49]:
# exp_with_range_reduction.py
import math
from typing import List, Tuple

LN2 = math.log(2.0)

def factorial_coeffs(n_terms: int) -> List[float]:
    coeffs = []
    f = 1.0
    for n in range(0, n_terms + 1):
        if n == 0:
            f = 1.0
        else:
            f *= n
        coeffs.append(1.0 / f)
    return coeffs

# -------------------------
# Horner core (float)
# -------------------------
def exp_horner_float_centered(q: float, coeffs: List[float]) -> float:
    """Evaluate e^{q} with Horner given coeffs = [1/0!, 1/1!, ..., 1/N!],
       where q is small (residual around expansion center)."""
    s = coeffs[-1]
    for c in reversed(coeffs[:-1]):
        s = c + q * s
    return s

# -------------------------
# Top-level - floating with range reduction
# -------------------------
def exp_range_float(x: float, n_terms: int = 6) -> float:
    """
    Compute exp(x) for arbitrary x using range reduction:
      x -> k, r  where k = floor(x / ln2), r = x - k*ln2
    Evaluate exp(r) around a = ln2/2 with Horner, then multiply by 2^k.
    """
    # range reduction
    k = math.floor(x / LN2)
    r = x - k * LN2

    # choose expansion center a in the middle of [0, ln2)
    a = LN2 / 2.0
    q = r - a  # small residual (should lie roughly within [-ln2/2, +ln2/2])

    coeffs = factorial_coeffs(n_terms)
    print("Coeffs: ", coeffs)
    e_q = exp_horner_float_centered(q, coeffs)  # evaluates e^{q} where q = r-a
    # result: e^x = 2^k * e^{r} = 2^k * e^{a} * e^{q}
    result = (2.0 ** k) * (math.exp(a) * e_q)
    return result

# -------------------------
# Fixed-point Q format core
# -------------------------
def exp_horner_fixed_q_core(q_q: int, coeffs_q: List[int], q_format: int) -> int:
    """
    Horner core in fixed-point (all values in Q(q_format)).
    Inputs:
      q_q       : q in Q format (integer)
      coeffs_q  : list of coefficients in Q format (integers)
    Returns s_q in Q format (integer)
    """
    s_q = coeffs_q[-1]
    for c_q in reversed(coeffs_q[:-1]):
        # multiply two Q numbers -> shift back by q_format
        prod = (q_q * s_q) >> q_format
        s_q = c_q + prod
    return s_q

def exp_range_fixed_q(x_q: int, q_format: int = 15, n_terms: int = 6) -> Tuple[int,int]:
    """
    Fixed-point version with range reduction.
      - x_q: integer representing x in Q(q_format)
    Returns:
      - y_q: integer representing approx(exp(x)) in Q(q_format)
      - k: integer power-of-two exponent: final result = 2^k * (y_q / 2^q_format)
    Note:
      - This function multiplies by 2^k via bit shifts. If k >= 0 -> left shift (may overflow),
        if k < 0 -> right shift.
      - No saturation or bit-width checks; Python big ints are used.
    """
    # compute k = floor(x / ln2) using floats for the division, then convert to int
    # (we could do this purely in fixed point, but float is simpler and acceptable for demo)
    x_float = x_q / (1 << q_format)
    k = math.floor(x_float / LN2)

    # r = x - k*ln2 in Q
    ln2_q = int(round(LN2 * (1 << q_format)))
    r_q = x_q - int(k) * ln2_q

    # center a = ln2/2 in Q
    a = LN2 / 2.0
    a_q = int(round(a * (1 << q_format)))

    q_q = r_q - a_q  # q in Q

    # prepare coeffs in Q
    coeffs_f = factorial_coeffs(n_terms)
    coeffs_q = [int(round(c * (1 << q_format))) for c in coeffs_f]

    # Horner core
    s_q = exp_horner_fixed_q_core(q_q, coeffs_q, q_format)

    # multiply by e^a (constant) in Q
    ea_q = int(round(math.exp(a) * (1 << q_format)))
    y_q = (ea_q * s_q) >> q_format  # still in Q

    # multiply by 2^k using shifts:
    if k >= 0:
        y_q_shifted = y_q << int(k)
    else:
        # right shift with arithmetic rounding toward zero
        y_q_shifted = y_q >> int(-k)

    return y_q_shifted, int(k)

# -------------------------
# Demo / small tests
# -------------------------
if __name__ == "__main__":
    test_xs = [-10.0, -3.0, -1.0, -0.5, 0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
    print("Floating-point range-reduced Horner vs math.exp(x)")
    for x in test_xs:
        approx = exp_range_float(x,n_terms=2)
        exact = math.exp(x)
        rel_err = (approx - exact) / exact
        print(f"x={x:6.2f} approx={approx:.6e} exact={exact:.6e} rel_err={rel_err:.3e}")

    # Fixed-point Q15 demo: show approx converted back to float
    Q = 15
    print("\nFixed-point Q15 demo (converted back to float) vs math.exp(x)")
    for x in [-3.0, -1.0, -0.5, 0.0, 0.1, 0.5, 1.0, 2.0]:
        x_q = int(round(x * (1 << Q)))
        y_q_shifted, k = exp_range_fixed_q(x_q, q_format=Q,n_terms=2)
        y_float = y_q_shifted / (1 << Q)
        print(f"x={x:6.2f} k={k:3d} approx_fixed={y_float:.6e} exact={math.exp(x):.6e} diff={y_float-math.exp(x):.3e}")


Floating-point range-reduced Horner vs math.exp(x)
Coeffs:  [1.0, 1.0, 0.5]
x=-10.00 approx=4.539898e-05 exact=4.539993e-05 rel_err=-2.083e-05
Coeffs:  [1.0, 1.0, 0.5]
x= -3.00 approx=4.977422e-02 exact=4.978707e-02 rel_err=-2.580e-04
Coeffs:  [1.0, 1.0, 0.5]
x= -1.00 approx=3.678757e-01 exact=3.678794e-01 rel_err=-1.014e-05
Coeffs:  [1.0, 1.0, 0.5]
x= -0.50 approx=6.069405e-01 exact=6.065307e-01 rel_err=6.756e-04
Coeffs:  [1.0, 1.0, 0.5]
x=  0.00 approx=1.009017e+00 exact=1.000000e+00 rel_err=9.017e-03
Coeffs:  [1.0, 1.0, 0.5]
x=  0.10 approx=1.108497e+00 exact=1.105171e+00 rel_err=3.009e-03
Coeffs:  [1.0, 1.0, 0.5]
x=  0.50 approx=1.647836e+00 exact=1.648721e+00 rel_err=-5.367e-04
Coeffs:  [1.0, 1.0, 0.5]
x=  1.00 approx=2.718311e+00 exact=2.718282e+00 rel_err=1.076e-05
Coeffs:  [1.0, 1.0, 0.5]
x=  2.00 approx=7.369817e+00 exact=7.389056e+00 rel_err=-2.604e-03
Coeffs:  [1.0, 1.0, 0.5]
x=  5.00 approx=1.486382e+02 exact=1.484132e+02 rel_err=1.516e-03
Coeffs:  [1.0, 1.0, 0.5]
x= 10.00 

In [5]:
def polynomial_horner(x, coeffs):
    """
    Calculates a + bx + cx^2 + dx^3 + ex^4 using Horner's Method.
    coeffs should be [a, b, c, d, e]
    """
    result = 0
    # Iterate through coefficients in reverse (e, d, c, b, a)
    for c in reversed(coeffs):
        result = result * x + c
        # print(f"Intermediate result: {result}")
    return result

# Example: a=1, b=1, c=0.5, d=0.16, e=0.04 (approximate e^x terms)
coefficients = [1, 1, 0.5, 0.1666667, 0.041666667]
x_val = 7

print(f"Result (Horner): {polynomial_horner(x_val, coefficients)}")

Result (Horner): 189.708345567
